In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from datetime import datetime

df = spark.table("pharma_catalog.bronze.bronze_inventory")
display(df)

In [0]:
# COMMAND ----------
# Doublons
df_dedup = df.dropDuplicates(["inventory_id"])
doublons = df.count() - df_dedup.count()
print(f"🔁 Doublons détectés : {doublons}")

# COMMAND ----------
df_quality = df_dedup.withColumn(
    "rule_failed",
    F.when(F.col("inventory_id").isNull(), "null_inventory_id")
    .when(F.col("product_id").isNull(), "null_product_id")
    .when(F.col("warehouse_id").isNull(), "null_warehouse_id")
    .when(F.col("batch_id").isNull(), "null_batch_id")
    .when(F.col("quantity_on_hand").isNull(), "null_quantity")
    .when(F.col("quantity_on_hand") < 0, "stock_negatif")
    .when(F.col("manufacturing_date").isNull(), "null_manufacturing_date")
    .when(F.col("expiry_date").isNull(), "null_expiry_date")
    .when(F.col("expiry_date") <= F.col("manufacturing_date"), "date_incoherente")
    .otherwise(None)
)

passed = df_quality.filter(F.col("rule_failed").isNull()).drop("rule_failed")
failed = df_quality.filter(F.col("rule_failed").isNotNull())

print(f" Lignes OK : {passed.count()}")
print(f" Lignes rejetées : {failed.count()}")

In [0]:
# COMMAND ----------
# Voir les lignes rejetées
display(failed)

In [0]:
# COMMAND ----------
if failed.count() > 0:
    df_quarantine = failed.withColumn("source_table", F.lit("bronze_inventory")) \
                           .withColumn("processing_date", F.lit(datetime.now().strftime("%Y-%m-%d")))

    df_quarantine.write.mode("append") \
                       .option("mergeSchema", "true") \
                       .saveAsTable("pharma_catalog.silver_quarantine.quarantine")
    print(f" {failed.count()} ligne(s) envoyée(s) en quarantine")
else:
    print(" Aucune ligne rejetée — quarantine vide")

In [0]:
# COMMAND ----------
# Envoi des lignes propres vers Silver
passed.write.mode("overwrite").saveAsTable("pharma_catalog.silver.silver_inventory")

print(f"✅ {passed.count()} ligne(s) chargée(s) dans silver_inventory")

In [0]:
# COMMAND ----------
display(spark.table("pharma_catalog.silver.silver_inventory"))

# COMMAND ----------
display(spark.table("pharma_catalog.silver_quarantine.quarantine"))